# AIMDL API Library Walkthrough

This notebook walks through the `aimdl_examples` Python library — the higher-level wrapper around the raw Girder API. It covers authentication, data type discovery, paginated downloads, client-side filtering, and result parsing.

For the raw endpoint reference (parameters, response shapes), see `01_api_endpoints.ipynb`.

## Imports

In [1]:
import io
import json
import sys
from pathlib import Path

import requests
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import (
    get_girder_client,
    download_item_to_disk,
    get_item_to_memory,
    fetch_and_parse,
    paginate_datafiles,
    get_output_dir,
)

## Authentication

`get_girder_client()` reads `GIRDER_API_URL` and `GIRDER_API_KEY` from the environment, creates a `GirderClient`, and verifies authentication in one call.

Passing a `requests.Session` lets all API calls share a single connection pool — important for throughput when downloading many files.

In [2]:
session = requests.Session()
gc = get_girder_client(session=session)
print('Connected to:', gc.urlBase)

Connected to: https://data.htmdec.org/api/v1/


## Inspecting the raw response

Every item returned by `aimdl/datafiles` has this core shape. The `meta` dict contains the IGSN (sample identifier), and provenance information if available. Additional custom metadata fields (e.g.,`meta.runId` for derived data processed using dagster) require `extraFields` to be included (see below).

In [3]:
sample_items = gc.get('aimdl/datafiles', parameters={
    'dataType': 'pdv_alpss_result',
    'limit': 1,
    'extraFields': '["meta.prov", "meta.runId"]',
})

example_item = sample_items[0]
print(json.dumps(example_item, indent=2))

{
  "_id": "6a296cb2ac4f2d98ffa04626",
  "_modelType": "item",
  "baseParentId": "665de536bcc722774ce53754",
  "baseParentType": "collection",
  "created": "2026-06-10T13:54:58.824000+00:00",
  "creatorId": "6a280b9d20783bdc81a10fc2",
  "folderId": "6a296cb2ac4f2d98ffa0460f",
  "meta": {
    "data_type": "pdv_alpss_result",
    "igsn": "JHAMAA00001-S5R4C3",
    "prov": {
      "wasDerivedFrom": "6894efd2d48b09febbb57e48",
      "wasGeneratedBy": "alpss_dagster/0.1.1 alpss/1.7.1"
    },
    "runId": "ac25c1d9-ebdf-4bdc-9ce1-b692c2e78a7e"
  },
  "name": "C1--20250807--00001-results.csv",
  "size": 1418
}


## Data types

The `dataType` parameter controls what category of files `aimdl/datafiles` returns. Use `aimdl/datatype` to get the authoritative list from the server, or reference this table:

| Data type | Experiment | Description |
|---|---|---|
| `pdv_trace` | HELIX | Raw PDV velocity trace files |
| `pdv_alpss_output` | HELIX | ALPSS analysis outputs (velocity curves, fits, plots) |
| `pdv_alpss_result` | HELIX | Compiled results CSV — one row per experiment |
| `pdv_experiment_log` | HELIX | Experiment metadata and settings |
| `xrd_raw` | MAXIMA | Raw XRD detector images |
| `xrd_calibrant_raw` | MAXIMA | Raw calibrant images |
| `xrd_derived` | MAXIMA | Processed data: intensity CSVs, scan images, TIFFs |
| `xrd_calibrant_derived` | MAXIMA | Processed calibrant data |
| `xrd_metadata` | MAXIMA | Metadata and logs |

In [ ]:
# Fetch the live list from the server
data_types = gc.get('aimdl/datatype')
print('Available data types:')
for dt in data_types:
    print(' ', dt)

Available data types:
[None, 'pdv_alpss_output', 'pdv_alpss_result', 'pdv_experiment_log', 'pdv_trace', 'xrd_calibrant_derived', 'xrd_calibrant_raw', 'xrd_derived', 'xrd_metadata', 'xrd_raw', 'xrf_raw']
  None
  pdv_alpss_output
  pdv_alpss_result
  pdv_experiment_log
  pdv_trace
  xrd_calibrant_derived
  xrd_calibrant_raw
  xrd_derived
  xrd_metadata
  xrd_raw
  xrf_raw


## Requesting extra metadata fields

By default, `aimdl/datafiles` returns only a subset of each item's fields. Pass `extraFields` as a JSON-encoded list of dotted paths to include additional metadata.

The most useful extras are `meta.prov` (provenance, including ALPSS versions) and `meta.runId` (the Dagster run that produced the file).

In [7]:
# Without extraFields — meta only contains igsn
without = gc.get('aimdl/datafiles', parameters={'dataType': 'pdv_alpss_result', 'limit': 1})
print('meta (default):', without[0].get('meta'))

print()

# With extraFields — meta also includes prov and runId
with_extras = gc.get('aimdl/datafiles', parameters={
    'dataType': 'pdv_alpss_result',
    'limit': 1,
    'extraFields': '["meta.prov", "meta.runId"]',
})
print('meta (with extras):', json.dumps(with_extras[0].get('meta'), indent=2))

meta (default): {'data_type': 'pdv_alpss_result', 'igsn': 'JHAMAA00001-S5R4C3'}

meta (with extras): {
  "data_type": "pdv_alpss_result",
  "igsn": "JHAMAA00001-S5R4C3",
  "prov": {
    "wasDerivedFrom": "6894efd2d48b09febbb57e48",
    "wasGeneratedBy": "alpss_dagster/0.1.1 alpss/1.7.1"
  },
  "runId": "ac25c1d9-ebdf-4bdc-9ce1-b692c2e78a7e"
}


## Paginating with `paginate_datafiles`

`paginate_datafiles` handles pagination (limit/offset looping) and parallelization (distributing work across a thread pool) automatically, returning a flat list of results. This lets you focus on your worker function without managing API pagination or threading.

```python
paginate_datafiles(
    gc,
    data_type,          # required: e.g. 'pdv_alpss_result'
    worker_fn,          # called once per item in a thread pool
    item_filter=None,   # optional client-side filter: item -> bool
    max_workers=8,      # thread pool size
    **query_params,     # forwarded to aimdl/datafiles
)
```

The pagination loop checks the raw batch size (before `item_filter`) to detect the last page — so filtering doesn't cause it to stop early.

In [8]:
# Collect all item names for one data type
names = paginate_datafiles(
    gc,
    'pdv_alpss_result',
    worker_fn=lambda item: item['name'],
    extraFields='["meta.runId"]',
)

print(f'Total pdv_alpss_result items: {len(names)}')
print('First 5:', names[:5])

Total pdv_alpss_result items: 3297
First 5: ['C1--20250807--00011-results.csv', 'C1--20250807--00032-results.csv', 'C1--20250807--00027-results.csv', 'C1--20250807--00024-results.csv', 'C1--20250807--00021-results.csv']


## Filtering items client-side

`item_filter` is a callable `(item) -> bool` applied to each item after it's fetched but before the worker runs. Use it when the server doesn't support the filter you need.

Common patterns:
- Filter by IGSN: `lambda item: item.get('meta', {}).get('igsn') == TARGET_IGSN`
- Filter by filename: `lambda item: 'velocity' in item.get('name', '')`

In [9]:
# Count pdv_alpss_output files whose name contains 'velocity'
velocity_files = paginate_datafiles(
    gc,
    'pdv_alpss_output',
    worker_fn=lambda item: item['name'],
    item_filter=lambda item: 'velocity' in item.get('name', ''),
)

print(f'velocity pdv_alpss_output files: {len(velocity_files)}')
print('Sample:', velocity_files[:3])

velocity pdv_alpss_output files: 6594
Sample: ['C1--20250807--00002-velocity--smooth.csv', 'C1--20250807--00004-velocity.csv', 'C1--20250807--00004-velocity--smooth.csv']


## Downloading to disk

`download_item_to_disk(gc, item, output_dir)` streams the file to disk and returns a metadata dict with `name`, `path`, `size`, and `itemId`.

`get_output_dir(item, base_dir)` builds a path that includes the item's IGSN as a subdirectory — so files from the same sample are grouped together.

In [10]:
# Download one pdv_alpss_result file as a demo
one_item = gc.get('aimdl/datafiles', parameters={'dataType': 'pdv_alpss_result', 'limit': 1})[0]

BASE_DIR = './downloads_demo'
output_dir = get_output_dir(one_item, BASE_DIR)
print('Output directory:', output_dir)

result = download_item_to_disk(gc, one_item, output_dir)
print('Downloaded:', result)

Output directory: ./downloads_demo/JHAMAA00001-S5R4C3
Downloaded: {'name': 'C1--20250807--00001-results.csv', 'path': 'downloads_demo/JHAMAA00001-S5R4C3/C1--20250807--00001-results.csv', 'size': 1418, 'itemId': '6a296cb2ac4f2d98ffa04626'}


## Streaming to memory

`get_item_to_memory(gc, item)` is a generator that yields raw bytes chunks. Wrap in `io.BytesIO` to get a file-like object you can pass to `pd.read_csv`, `PIL.Image.open`, a neural network, etc.

In [11]:
content = io.BytesIO(b''.join(get_item_to_memory(gc, one_item)))
print(f'Downloaded {content.getbuffer().nbytes:,} bytes into memory')

# Peek at the first line
content.seek(0)
print('First line:', content.readline().decode().strip())

Downloaded 1,418 bytes into memory
First line: Date,Time,File Name,Run Time,Error Message,Velocity OK,Velocity at Max Compression,Time at Max Compression,Velocity at Max Compression Freq Uncertainty,Velocity at Max Compression Vel Uncertainty,Carrier Frequency,Spect Time Res,Spect Freq Res,Spect Velocity Res,Signal Start Time,Smoothing Characteristic Time,Spall Enabled,Spall OK,Velocity at Max Tension,Time at Max Tension,Velocity at Max Tension Freq Uncertainty,Velocity at Max Tension Vel Uncertainty,Velocity at Recompression,Time at Recompression,Spall Strength,Strain Rate,Uncertainty OK,Spall Strength Uncertainty,Strain Rate Uncertainty,HEL Enabled,HEL OK,HEL Strength (GPa),HEL Uncertainty (GPa),HEL Free Surface Velocity (m/s),HEL Time Detection (ns),HEL Consecutive Points,HEL Segment Duration (ns),HEL Strain Rate,Peak Shock Stress


## Parsing ALPSS results

`fetch_and_parse(gc, item)` combines the download and CSV parse in one call. It returns a single dict representing one experiment, with metadata fields merged in from the Girder item.

For bulk analysis, use `paginate_datafiles` with `fetch_and_parse` as the worker to build a list of dicts, then convert to a DataFrame.

In [12]:
# Need extraFields for prov/runId — fetch_and_parse reads them from item.meta
result_item = gc.get('aimdl/datafiles', parameters={
    'dataType': 'pdv_alpss_result',
    'limit': 1,
    'extraFields': '["meta.prov", "meta.runId"]',
})[0]

parsed = fetch_and_parse(gc, result_item)

print('Keys in parsed result:')
print(list(parsed.keys()))
print()
print('Metadata fields added by parse_results:')
for key in ('igsn', 'itemId', 'runId', 'alpss_version', 'alpss_dagster_version'):
    print(f'  {key}: {parsed.get(key)}')

Keys in parsed result:
['Date', 'File Name', 'Run Time', 'Error Message', 'Velocity OK', 'Velocity at Max Compression', 'Time at Max Compression', 'Velocity at Max Compression Freq Uncertainty', 'Velocity at Max Compression Vel Uncertainty', 'Carrier Frequency', 'Spect Time Res', 'Spect Freq Res', 'Spect Velocity Res', 'Signal Start Time', 'Smoothing Characteristic Time', 'Spall Enabled', 'Spall OK', 'Velocity at Max Tension', 'Time at Max Tension', 'Velocity at Max Tension Freq Uncertainty', 'Velocity at Max Tension Vel Uncertainty', 'Velocity at Recompression', 'Time at Recompression', 'Spall Strength', 'Strain Rate', 'Uncertainty OK', 'Spall Strength Uncertainty', 'Strain Rate Uncertainty', 'HEL Enabled', 'HEL OK', 'HEL Strength (GPa)', 'HEL Uncertainty (GPa)', 'HEL Free Surface Velocity (m/s)', 'HEL Time Detection (ns)', 'HEL Consecutive Points', 'HEL Segment Duration (ns)', 'HEL Strain Rate', 'Peak Shock Stress', 'igsn', 'itemId', 'runId', 'alpss_dagster_version', 'alpss_version']